In [1]:
import pandas as pd
import numpy as np
import re
import ast

pd.set_option('display.max_columns', None)

BASE_DIR = r'E:\DATA ANAYLST PROJECTS\Real estate\data'

CITY_FILES = {
    'Gurgaon': BASE_DIR + r'\gurgaon_10k.csv',
    'Hyderabad': BASE_DIR + r'\hyderabad.csv',
    'Kolkata': BASE_DIR + r'\kolkata.csv',
    'Mumbai': BASE_DIR + r'\mumbai.csv',
}

FACET_DIR = BASE_DIR + r'\facets\facets\\'

print("Setup done.")

Setup done.


In [2]:
def load_map(filename):
    """Read a facet CSV and turn it into a {code: label} dictionary."""
    df = pd.read_csv(FACET_DIR + filename)
    d = df.set_index('id')['label'].to_dict()
    out = {}
    for k, v in d.items():
        try:
            out[int(k)] = v
        except ValueError:
            out[k] = v
    return out

age_map = load_map('AGE.csv')
furnish_map = load_map('FURNISH.csv')
facing_map = load_map('FACING_DIRECTION.csv')
owntype_map = load_map('OWNERSHIP_TYPE.csv')

amenities_df = pd.read_csv(FACET_DIR + 'AMENITIES.csv')

# amenities we'll track as Yes/No flags in the dashboard (the most dashboard-useful ones)
KEY_AMENITIES = {
    'Swimming Pool': '1', 'Club house / Community Center': '3', 'Park': '6',
    'Security Personnel': '9', 'Fitness Centre / GYM': '12', 'Lift(s)': '21',
    'CCTV Surveillance': '36', 'Power Back-up': '2',
}

print("AGE codes:", age_map)
print("FURNISH codes:", furnish_map)
print("FACING codes:", facing_map)
print("OWNERSHIP codes:", owntype_map)

AGE codes: {1: '1-5 Year Old Property', 2: '5-10 Year Old Property', 3: '10+ Year Old Property', 5: 'Under Construction', 6: '0-1 Year Old Property'}
FURNISH codes: {1: 'Furnished', 2: 'Unfurnished', 4: 'Semifurnished'}
FACING codes: {1: 'North', 2: 'South', 3: 'East', 4: 'West', 5: 'North-East', 6: 'North-West', 7: 'South-East', 8: 'South-West'}
OWNERSHIP codes: {1: 'Freehold', 2: 'Leasehold', 3: 'Co-operative Society', 4: 'Power of Attorney'}


In [14]:
def parse_price(val):
    """'2.63 Cr' -> 26300000 | '69.25 L' -> 6925000 | '1.5 - 5.99 L' -> avg of range
    '20,000/Bedroom' -> 20000 (per-bed rent, kept as-is) | 'Price on Request' -> NaN"""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if 'request' in s.lower():
        return np.nan
    if '/' in s:
        num_part = s.split('/')[0].replace(',', '').strip()
        try:
            return float(num_part)
        except ValueError:
            return np.nan
    s_clean = s.replace(',', '')
    unit = 'Cr' if 'Cr' in s_clean else ('L' if 'L' in s_clean else None)
    numbers = re.findall(r'\d+\.?\d*', s_clean)
    numbers = [float(n) for n in numbers if n]
    if not numbers:
        return np.nan
    avg = sum(numbers) / len(numbers)
    if unit == 'Cr':
        return avg * 1e7
    elif unit == 'L':
        return avg * 1e5
    return avg

def parse_area(val):
    """'3434 sq.ft.' -> 3434.0 | '1155-1395 sq.ft.' -> 1275.0 (avg of range)"""
    if pd.isna(val):
        return np.nan
    s = str(val).replace(',', '')
    numbers = re.findall(r'\d+\.?\d*', s)
    numbers = [float(n) for n in numbers if n]
    if not numbers:
        return np.nan
    return sum(numbers) / len(numbers)

def get_listing_category(transact_type, property_type=None):
    """1.0 -> Resale | 2.0 -> New Booking
    NaN -> PG/Shared Rental, UNLESS it's a Land listing, which is just an Unlabeled Sale"""
    if pd.isna(transact_type):
        if property_type == 'Residential Land':
            return 'Land Sale (Unlabeled)'
        return 'PG/Shared Rental'
    elif transact_type == 1.0:
        return 'Resale'
    elif transact_type == 2.0:
        return 'New Booking'
    return 'Unknown'

def parse_location_dict(val):
    """The 'location' column is a Python-dict-style string like
    "{'CITY_NAME': 'Gurgaon', 'LOCALITY_NAME': 'Sector 84', ...}" -- pull out the two fields we need."""
    if pd.isna(val):
        return pd.Series([np.nan, np.nan])
    try:
        d = ast.literal_eval(val)
        return pd.Series([d.get('LOCALITY_NAME'), d.get('CITY_NAME')])
    except (ValueError, SyntaxError):
        return pd.Series([np.nan, np.nan])

def clean_floor_num(val):
    """'G' (ground floor) -> 0 | '14' -> 14 | '1.0' -> 1"""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    if s == 'G':
        return 0
    try:
        return int(float(s))
    except ValueError:
        return np.nan

def count_amenities(val):
    """Count how many amenity codes are listed for a property."""
    if pd.isna(val):
        return 0
    codes = [c.strip() for c in str(val).split(',') if c.strip()]
    return len(codes)

def has_amenity(val, amenity_code):
    """Check whether a specific amenity code is present."""
    if pd.isna(val):
        return False
    codes = [c.strip() for c in str(val).split(',')]
    return amenity_code in codes

print("Functions defined.")

Functions defined.


In [15]:
WANTED_COLS = ['PROP_ID', 'PROPERTY_TYPE', 'CITY', 'TRANSACT_TYPE', 'OWNTYPE',
               'BEDROOM_NUM', 'BATHROOM_NUM', 'BALCONY_NUM', 'FURNISH', 'FACING',
               'AGE', 'FLOOR_NUM', 'TOTAL_FLOOR', 'AREA', 'PRICE', 'PRICE_SQFT',
               'PROP_NAME', 'SOCIETY_NAME', 'BUILDING_NAME', 'location', 'AMENITIES']

all_dfs = []

for city_label, path in CITY_FILES.items():
    cols_present = pd.read_csv(path, nrows=0).columns.tolist()
    use_cols = [c for c in WANTED_COLS if c in cols_present]
    df = pd.read_csv(path, usecols=use_cols, on_bad_lines='skip', engine='python')

    # add any missing wanted columns as blank, so all 4 city frames line up
    for c in WANTED_COLS:
        if c not in df.columns:
            df[c] = np.nan

    # --- apply cleaning ---
    df['PRICE_INR'] = df['PRICE'].apply(parse_price)
    df['AREA_SQFT'] = df['AREA'].apply(parse_area)
    df['LISTING_CATEGORY'] = df.apply(lambda r: get_listing_category(r['TRANSACT_TYPE'], r['PROPERTY_TYPE']), axis=1)
    df['AGE_LABEL'] = df['AGE'].map(age_map)
    df['FURNISH_LABEL'] = df['FURNISH'].map(furnish_map)
    df['FACING_LABEL'] = df['FACING'].map(facing_map)
    df['OWNERSHIP_LABEL'] = df['OWNTYPE'].map(owntype_map)
    df['FLOOR_NUM_CLEAN'] = df['FLOOR_NUM'].apply(clean_floor_num)
    df['AMENITIES_COUNT'] = df['AMENITIES'].apply(count_amenities)

    for amenity_name, code in KEY_AMENITIES.items():
        col_name = 'HAS_' + re.sub(r'[^A-Z0-9]+', '_', amenity_name.upper()).strip('_')
        df[col_name] = df['AMENITIES'].apply(lambda v: has_amenity(v, code))

    loc_parsed = df['location'].apply(parse_location_dict)
    loc_parsed.columns = ['LOCALITY', 'CITY_NAME']
    df = pd.concat([df, loc_parsed], axis=1)

    # recalculate price-per-sqft from our own cleaned numbers (don't trust the original column)
    df['PRICE_PER_SQFT_CALC'] = np.where(
        (df['AREA_SQFT'] > 0) & df['PRICE_INR'].notna() & (df['LISTING_CATEGORY'] != 'PG/Shared Rental'),
        df['PRICE_INR'] / df['AREA_SQFT'],
        np.nan
    )

    df['SOURCE_CITY'] = city_label
    all_dfs.append(df)
    print(f"{city_label}: {df.shape[0]} rows processed")

combined = pd.concat(all_dfs, ignore_index=True)
print("\nCombined shape:", combined.shape)

Gurgaon: 10704 rows processed
Hyderabad: 9487 rows processed
Kolkata: 8797 rows processed
Mumbai: 9514 rows processed

Combined shape: (38502, 42)


In [16]:
FINAL_COLS = [
    'PROP_ID', 'SOURCE_CITY', 'CITY_NAME', 'LOCALITY', 'PROPERTY_TYPE',
    'LISTING_CATEGORY', 'OWNERSHIP_LABEL', 'BEDROOM_NUM', 'BATHROOM_NUM',
    'BALCONY_NUM', 'FURNISH_LABEL', 'FACING_LABEL', 'AGE_LABEL',
    'FLOOR_NUM_CLEAN', 'TOTAL_FLOOR', 'AREA_SQFT', 'PRICE_INR',
    'PRICE_PER_SQFT_CALC', 'PROP_NAME', 'SOCIETY_NAME', 'BUILDING_NAME',
    'AMENITIES_COUNT'
] + [c for c in combined.columns if c.startswith('HAS_')]

final_df = combined[FINAL_COLS].copy()

# drop exact duplicate listings (same PROP_ID appearing more than once)
before = len(final_df)
final_df = final_df.drop_duplicates(subset=['PROP_ID'])
print(f"Dropped {before - len(final_df)} duplicate PROP_IDs")

print("\nFinal shape:", final_df.shape)
print("\nNulls per column:")
print(final_df.isna().sum())

print("\nListing category breakdown:")
print(final_df['LISTING_CATEGORY'].value_counts())

print("\nRows per city:")
print(final_df['SOURCE_CITY'].value_counts())


Dropped 15 duplicate PROP_IDs

Final shape: (38487, 30)

Nulls per column:
PROP_ID                                0
SOURCE_CITY                            0
CITY_NAME                              0
LOCALITY                               0
PROPERTY_TYPE                          0
LISTING_CATEGORY                       0
OWNERSHIP_LABEL                     6479
BEDROOM_NUM                         3554
BATHROOM_NUM                       27783
BALCONY_NUM                         9997
FURNISH_LABEL                      10879
FACING_LABEL                       10649
AGE_LABEL                           3542
FLOOR_NUM_CLEAN                     6591
TOTAL_FLOOR                          652
AREA_SQFT                              0
PRICE_INR                            395
PRICE_PER_SQFT_CALC                 6879
PROP_NAME                           5018
SOCIETY_NAME                        5018
BUILDING_NAME                       5031
AMENITIES_COUNT                        0
HAS_SWIMMING_POOL      

In [17]:
print("PRICE_INR stats:")
print(final_df['PRICE_INR'].describe())

print("\nAREA_SQFT stats:")
print(final_df['AREA_SQFT'].describe())

print("\nPRICE_PER_SQFT_CALC stats:")
print(final_df['PRICE_PER_SQFT_CALC'].describe())

print("\nBEDROOM_NUM value counts:")
print(final_df['BEDROOM_NUM'].value_counts().sort_index())

PRICE_INR stats:
count    3.809200e+04
mean     2.316867e+07
std      5.840353e+07
min      5.000000e+02
25%      3.200000e+06
50%      1.050000e+07
75%      2.350000e+07
max      3.000000e+09
Name: PRICE_INR, dtype: float64

AREA_SQFT stats:
count    3.848700e+04
mean     3.580166e+03
std      7.894942e+04
min      1.000000e+00
25%      9.315000e+02
50%      1.440000e+03
75%      2.100000e+03
max      7.840800e+06
Name: AREA_SQFT, dtype: float64

PRICE_PER_SQFT_CALC stats:
count    3.160800e+04
mean     2.010353e+04
std      1.521248e+05
min      1.111242e-01
25%      5.524862e+03
50%      9.259259e+03
75%      1.530612e+04
max      1.333333e+07
Name: PRICE_PER_SQFT_CALC, dtype: float64

BEDROOM_NUM value counts:
BEDROOM_NUM
1.0      3096
2.0     10495
3.0     14252
4.0      5771
5.0       815
6.0       190
7.0        61
8.0        45
9.0        63
10.0       53
11.0        7
12.0       26
14.0        5
15.0        4
16.0        6
17.0        1
18.0        6
19.0        2
20.0        

In [18]:
# Sensible real-world thresholds for INDIAN residential real estate
# (loose enough to keep legitimate luxury/farmhouse listings, tight enough to catch garbage data)
final_df['IS_OUTLIER'] = (
    (final_df['PRICE_INR'] < 50000) |              # less than 50k total price is not a real sale price
    (final_df['PRICE_INR'] > 5e8) |                 # more than 50 crore is extremely rare, likely data error for this dataset
    (final_df['AREA_SQFT'] < 100) |                  # smaller than 100 sqft is not a real livable unit
    (final_df['AREA_SQFT'] > 50000) |                # larger than 50,000 sqft is not a typical flat/house (could be land mislabeled)
    (final_df['BEDROOM_NUM'] > 10)                   # more than 10 BHK is not a typical residential unit
)

print("Outlier rows flagged:", final_df['IS_OUTLIER'].sum())
print("Out of total:", len(final_df))
print(f"That's {final_df['IS_OUTLIER'].mean()*100:.2f}% of the data")

print("\nWhat do flagged rows look like?")
print(final_df[final_df['IS_OUTLIER']][['PROPERTY_TYPE','BEDROOM_NUM','AREA_SQFT','PRICE_INR','LISTING_CATEGORY']].head(15))

Outlier rows flagged: 4680
Out of total: 38487
That's 12.16% of the data

What do flagged rows look like?
                 PROPERTY_TYPE  BEDROOM_NUM  AREA_SQFT    PRICE_INR  \
4        Residential Apartment          3.0     2290.0      40000.0   
6        Residential Apartment          4.0     3150.0      45000.0   
36       Residential Apartment          2.0     1330.0      43500.0   
74       Residential Apartment          3.0     2450.0      45000.0   
101    Independent House/Villa         19.0      512.0  103500000.0   
102  Independent/Builder Floor          2.0     1440.0      35000.0   
110      Residential Apartment          3.0     1800.0      29000.0   
111      Residential Apartment          3.0     1800.0      39000.0   
112      Residential Apartment          3.0     1185.0      19200.0   
125      Residential Apartment          3.0     1852.0      33000.0   
129           Residential Land          NaN   784080.0  300000000.0   
137      Residential Apartment          2.

In [19]:
def flag_outlier(row):
    cat = row['LISTING_CATEGORY']
    price = row['PRICE_INR']
    area = row['AREA_SQFT']
    beds = row['BEDROOM_NUM']
    ptype = row['PROPERTY_TYPE']

    # Land/plot listings: area can legitimately be huge, skip area checks, just check beds make no sense for land
    if ptype == 'Residential Land':
        return pd.isna(price) or price < 10000

    # PG/Shared rentals: price is per-bed monthly rent, much smaller numbers, area is the room/flat area
    if cat == 'PG/Shared Rental':
        return (
            (pd.notna(price) and (price < 1000 or price > 100000)) or   # per-bed rent realistically 1k-100k/month
            (pd.notna(area) and (area < 50 or area > 10000)) or
            (pd.notna(beds) and beds > 10)
        )

    # Resale / New Booking: full flat/house sale price
    return (
        (pd.notna(price) and (price < 50000 or price > 5e8)) or
        (pd.notna(area) and (area < 100 or area > 50000)) or
        (pd.notna(beds) and beds > 10)
    )

final_df['IS_OUTLIER'] = final_df.apply(flag_outlier, axis=1)

print("Outlier rows flagged:", final_df['IS_OUTLIER'].sum())
print("Out of total:", len(final_df))
print(f"That's {final_df['IS_OUTLIER'].mean()*100:.2f}% of the data")

print("\nWhat do flagged rows look like now?")
print(final_df[final_df['IS_OUTLIER']][['PROPERTY_TYPE','BEDROOM_NUM','AREA_SQFT','PRICE_INR','LISTING_CATEGORY']].head(15))

Outlier rows flagged: 799
Out of total: 38487
That's 2.08% of the data

What do flagged rows look like now?
                 PROPERTY_TYPE  BEDROOM_NUM  AREA_SQFT    PRICE_INR  \
48       Residential Apartment          4.0     4280.0     250000.0   
70       Residential Apartment          4.0     3104.0     150000.0   
71       Residential Apartment          4.0     7673.0     250000.0   
72       Residential Apartment          3.0     2875.0     238000.0   
73       Residential Apartment          3.0     2200.0     199000.0   
75       Residential Apartment          3.0     2150.0     165000.0   
78       Residential Apartment          4.0     3390.0     155000.0   
79       Residential Apartment          3.0     2300.0     150000.0   
97   Independent/Builder Floor          4.0     2700.0     150000.0   
101    Independent House/Villa         19.0      512.0  103500000.0   
116      Residential Apartment          4.0     3160.0     160000.0   
117      Residential Apartment          

In [20]:
def flag_outlier(row):
    cat = row['LISTING_CATEGORY']
    price = row['PRICE_INR']
    area = row['AREA_SQFT']
    beds = row['BEDROOM_NUM']
    ptype = row['PROPERTY_TYPE']

    if ptype == 'Residential Land':
        return pd.isna(price) or price < 10000

    if cat == 'PG/Shared Rental':
        # normalize by bedroom count where available, else just sanity-check raw price
        if pd.notna(beds) and beds > 0:
            price_per_bed = price / beds if pd.notna(price) else np.nan
            return (
                (pd.notna(price_per_bed) and (price_per_bed < 1000 or price_per_bed > 60000)) or
                (pd.notna(area) and (area < 50 or area > 15000)) or
                (beds > 10)
            )
        return pd.notna(price) and (price < 1000 or price > 300000)

    # Resale / New Booking: full flat/house sale price
    return (
        (pd.notna(price) and (price < 50000 or price > 5e8)) or
        (pd.notna(area) and (area < 100 or area > 50000)) or
        (pd.notna(beds) and beds > 10)
    )

final_df['IS_OUTLIER'] = final_df.apply(flag_outlier, axis=1)

print("Outlier rows flagged:", final_df['IS_OUTLIER'].sum())
print(f"That's {final_df['IS_OUTLIER'].mean()*100:.2f}% of the data")

print("\nFlagged rows breakdown by category:")
print(final_df[final_df['IS_OUTLIER']]['LISTING_CATEGORY'].value_counts())

print("\nSample flagged rows:")
print(final_df[final_df['IS_OUTLIER']][['PROPERTY_TYPE','BEDROOM_NUM','AREA_SQFT','PRICE_INR','LISTING_CATEGORY']].sample(15, random_state=1))

Outlier rows flagged: 464
That's 1.21% of the data

Flagged rows breakdown by category:
LISTING_CATEGORY
PG/Shared Rental    314
Resale              118
New Booking          32
Name: count, dtype: int64

Sample flagged rows:
                 PROPERTY_TYPE  BEDROOM_NUM  AREA_SQFT    PRICE_INR  \
24368         Residential Land          NaN    20000.0          NaN   
26890    Residential Apartment          8.0     1000.0       2900.0   
37311    Residential Apartment          2.0      900.0     130000.0   
26959  Independent House/Villa          5.0     1500.0       2500.0   
9545   Independent House/Villa         22.0     5400.0   54500000.0   
27072  Independent House/Villa          3.0     1500.0       2000.0   
9381   Independent House/Villa         11.0      540.0   24000000.0   
27067  Independent House/Villa          3.0      350.0       2500.0   
18205         Residential Land          NaN     5381.0          NaN   
36119    Residential Apartment          4.0     2300.0     450000

In [21]:
# Quick before/after comparison on the cleaned (non-outlier) data
clean_only = final_df[~final_df['IS_OUTLIER']]
print("Clean rows:", len(clean_only), "out of", len(final_df))

print("\nPRICE_INR stats (excluding outliers):")
print(clean_only.groupby('LISTING_CATEGORY')['PRICE_INR'].describe())

# Export — keep ALL rows (including flagged ones) but with the IS_OUTLIER column,
# so the dashboard tool (Power BI / Excel) can filter them out without losing data
output_path = r'E:\DATA ANAYLST PROJECTS\Real estate\combined_real_estate_cleaned.xlsx'
final_df.to_excel(output_path, index=False)
print(f"\nSaved to: {output_path}")
print("Final shape:", final_df.shape)

Clean rows: 38023 out of 38487

PRICE_INR stats (excluding outliers):
                         count          mean           std        min  \
LISTING_CATEGORY                                                        
Land Sale (Unlabeled)      9.0  5.044000e+06  3.573298e+06  1350000.0   
New Booking             6209.0  4.233096e+07  9.260169e+07   200000.0   
PG/Shared Rental        6170.0  3.977333e+04  3.782448e+04     1500.0   
Resale                 25271.0  2.335654e+07  4.258721e+07   100000.0   

                             25%         50%         75%           max  
LISTING_CATEGORY                                                        
Land Sale (Unlabeled)  3000000.0   4000000.0   5200000.0  1.200000e+07  
New Booking            7101000.0  15200000.0  30000000.0  4.996000e+08  
PG/Shared Rental         13000.0     27500.0     55000.0  3.000000e+05  
Resale                 6100000.0  14000000.0  26500000.0  3.000000e+09  

Saved to: E:\DATA ANAYLST PROJECTS\Real estate\combi

In [22]:
def flag_outlier(row):
    cat = row['LISTING_CATEGORY']
    price = row['PRICE_INR']
    area = row['AREA_SQFT']
    beds = row['BEDROOM_NUM']
    ptype = row['PROPERTY_TYPE']

    if ptype == 'Residential Land':
        # land can be large and pricey, but cap at a sane ceiling for this dataset too
        return pd.isna(price) or price < 10000 or price > 5e8

    if cat == 'PG/Shared Rental':
        if pd.notna(beds) and beds > 0:
            price_per_bed = price / beds if pd.notna(price) else np.nan
            return (
                (pd.notna(price_per_bed) and (price_per_bed < 1000 or price_per_bed > 60000)) or
                (pd.notna(area) and (area < 50 or area > 15000)) or
                (beds > 10)
            )
        # no bedroom count to normalize by -- use a tighter absolute ceiling
        return pd.notna(price) and (price < 1000 or price > 150000)

    return (
        (pd.notna(price) and (price < 50000 or price > 5e8)) or
        (pd.notna(area) and (area < 100 or area > 50000)) or
        (pd.notna(beds) and beds > 10)
    )

final_df['IS_OUTLIER'] = final_df.apply(flag_outlier, axis=1)

print("Outlier rows flagged:", final_df['IS_OUTLIER'].sum())
print(f"That's {final_df['IS_OUTLIER'].mean()*100:.2f}% of the data")

Outlier rows flagged: 475
That's 1.23% of the data


In [23]:
clean_only = final_df[~final_df['IS_OUTLIER']]
print("Clean rows:", len(clean_only), "out of", len(final_df))

print("\nPRICE_INR stats (excluding outliers):")
print(clean_only.groupby('LISTING_CATEGORY')['PRICE_INR'].describe())

output_path = r'E:\DATA ANAYLST PROJECTS\Real estate\combined_real_estate_cleaned.xlsx'
final_df.to_excel(output_path, index=False)
print(f"\nSaved to: {output_path}")
print("Final shape:", final_df.shape)

Clean rows: 38012 out of 38487

PRICE_INR stats (excluding outliers):
                         count          mean           std        min  \
LISTING_CATEGORY                                                        
Land Sale (Unlabeled)      9.0  5.044000e+06  3.573298e+06  1350000.0   
New Booking             6209.0  4.233096e+07  9.260169e+07   200000.0   
PG/Shared Rental        6169.0  3.975060e+04  3.778538e+04     1500.0   
Resale                 25261.0  2.290178e+07  3.294398e+07   100000.0   

                             25%         50%         75%          max  
LISTING_CATEGORY                                                       
Land Sale (Unlabeled)  3000000.0   4000000.0   5200000.0   12000000.0  
New Booking            7101000.0  15200000.0  30000000.0  499600000.0  
PG/Shared Rental         13000.0     27500.0     55000.0     300000.0  
Resale                 6100000.0  14000000.0  26500000.0  500000000.0  

Saved to: E:\DATA ANAYLST PROJECTS\Real estate\combined_re

In [24]:
pg = final_df[(final_df['LISTING_CATEGORY'] == 'PG/Shared Rental') & (~final_df['IS_OUTLIER'])]
print(pg.sort_values('PRICE_INR', ascending=False)[['PROPERTY_TYPE','BEDROOM_NUM','AREA_SQFT','PRICE_INR']].head(10))

                 PROPERTY_TYPE  BEDROOM_NUM  AREA_SQFT  PRICE_INR
17707  Independent House/Villa          5.0     7200.0   300000.0
13798  Independent House/Villa          5.0     4500.0   300000.0
27733    Residential Apartment          5.0     4500.0   280000.0
36824    Residential Apartment          6.0     4010.0   251000.0
18425  Independent House/Villa          5.0     4600.0   250000.0
1544     Residential Apartment          5.0     4650.0   250000.0
1407   Independent House/Villa          5.0     3150.0   240000.0
15281    Residential Apartment          4.0     4070.0   240000.0
448      Residential Apartment          4.0     4200.0   240000.0
33449    Residential Apartment          4.0     2200.0   239000.0


In [25]:
clean_only = final_df[~final_df['IS_OUTLIER']]
print("Clean rows:", len(clean_only), "out of", len(final_df))

print("\nPRICE_INR stats (excluding outliers):")
print(clean_only.groupby('LISTING_CATEGORY')['PRICE_INR'].describe())

output_path = r'E:\DATA ANAYLST PROJECTS\Real estate\combined_real_estate_cleaned.xlsx'
final_df.to_excel(output_path, index=False)
print(f"\nSaved to: {output_path}")
print("Final shape:", final_df.shape)

Clean rows: 38012 out of 38487

PRICE_INR stats (excluding outliers):
                         count          mean           std        min  \
LISTING_CATEGORY                                                        
Land Sale (Unlabeled)      9.0  5.044000e+06  3.573298e+06  1350000.0   
New Booking             6209.0  4.233096e+07  9.260169e+07   200000.0   
PG/Shared Rental        6169.0  3.975060e+04  3.778538e+04     1500.0   
Resale                 25261.0  2.290178e+07  3.294398e+07   100000.0   

                             25%         50%         75%          max  
LISTING_CATEGORY                                                       
Land Sale (Unlabeled)  3000000.0   4000000.0   5200000.0   12000000.0  
New Booking            7101000.0  15200000.0  30000000.0  499600000.0  
PG/Shared Rental         13000.0     27500.0     55000.0     300000.0  
Resale                 6100000.0  14000000.0  26500000.0  500000000.0  

Saved to: E:\DATA ANAYLST PROJECTS\Real estate\combined_re

In [26]:
# Filter to just Resale + New Booking, drop outliers, keep dashboard-relevant columns
dashboard_df = final_df[
    (final_df['LISTING_CATEGORY'].isin(['Resale', 'New Booking'])) &
    (~final_df['IS_OUTLIER'])
].copy()

print("Dashboard-ready rows:", len(dashboard_df))
print(dashboard_df['LISTING_CATEGORY'].value_counts())
print(dashboard_df['SOURCE_CITY'].value_counts())

# Drop columns not needed for this dashboard (PG/land-specific noise removed by the filter already)
dashboard_df = dashboard_df.drop(columns=['IS_OUTLIER'])

output_path2 = r'E:\DATA ANAYLST PROJECTS\Real estate\dashboard_data.xlsx'
dashboard_df.to_excel(output_path2, index=False, sheet_name='RawData')
print(f"\nSaved dashboard-ready file to: {output_path2}")
print("Final shape:", dashboard_df.shape)

Dashboard-ready rows: 31827
LISTING_CATEGORY
Resale         25305
New Booking     6522
Name: count, dtype: int64
SOURCE_CITY
Gurgaon      10089
Hyderabad     7857
Kolkata       7032
Mumbai        6849
Name: count, dtype: int64

Saved dashboard-ready file to: E:\DATA ANAYLST PROJECTS\Real estate\dashboard_data.xlsx
Final shape: (31827, 30)
